In [ ]:
from pathlib import Path

# Run this baseline for the attached SemanticCloneBench V3 Kaggle dataset.
# Add the SemanticCloneBench V3 dataset to the Kaggle notebook inputs before running all cells.
DATASET_KEYS = ("semantic-clone-bench",)
RUN_LABEL = "deckard_baseline"


# Kaggle's current PyTorch build cannot execute kernels on Tesla P100 (sm_60).
# These notebooks are intended for a T4-class accelerator; two T4s are fine,
# although this single-process implementation uses GPU 0.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is enabled. In Kaggle select GPU accelerator: 2x T4.")
_GPU_CAPABILITY = torch.cuda.get_device_capability(0)
_GPU_NAME = torch.cuda.get_device_name(0)
if _GPU_CAPABILITY[0] < 7:
    raise RuntimeError(
        f"{_GPU_NAME} has unsupported CUDA capability sm_{_GPU_CAPABILITY[0]}{_GPU_CAPABILITY[1]}. "
        "Select 2x T4 in Kaggle Accelerator settings, restart the session, and Run All."
    )
print({"gpu": _GPU_NAME, "capability": f"sm_{_GPU_CAPABILITY[0]}{_GPU_CAPABILITY[1]}", "gpu_count": torch.cuda.device_count()})
# Runtime profile. Use quick_1h for preliminary results; change only this
# value to extended_6_7h for the larger follow-up run.
RUN_PROFILE = "final_full"
RUN_PRESETS = {"quick_1h": {'max_train_pairs': 75000, 'max_valid_pairs': 10000, 'max_test_pairs': 10000}, "extended_6_7h": {'max_train_pairs': 100000, 'max_valid_pairs': 20000, 'max_test_pairs': 20000}}
# Use this profile in every method notebook for a data-equal comparison.
RUN_PRESETS["comparison_50k"] = {
    **RUN_PRESETS["quick_1h"],
    "max_train_pairs": 50_000,
    "max_valid_pairs": 10_000,
    "max_test_pairs": 10_000,
}

# Final paper protocol: use every available pair in each official split.
RUN_PRESETS["final_full"] = {
    **RUN_PRESETS["quick_1h"],
    "max_train_pairs": None,
    "max_valid_pairs": None,
    "max_test_pairs": None,
}

# --- bounded run budget (scripts/patch_kaggle_run_budget.py) ---
# Kaggle sessions are capped, and a run that dies at the limit produces nothing.
# Training data stays large so results remain comparable with the published
# table; validation and test are capped because a bigger validation split only
# sharpens one threshold, and a bigger test split only tightens an error bar we
# do not report.
RUN_PRESETS["bounded_10h"] = {
    **RUN_PRESETS["comparison_50k"],
    "max_train_pairs": 200_000,
    "max_valid_pairs": 20_000,
    "max_test_pairs": 20_000,
}

if RUN_PROFILE not in RUN_PRESETS:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}; choose one of {tuple(RUN_PRESETS)}")
RUN_CONFIG = RUN_PRESETS[RUN_PROFILE]


In [ ]:
# === per-language breakdown helper (patched by scripts/patch_kaggle_language_breakdown.py) ===
# Splits an already-computed set of test predictions by the language of each
# pair. No retraining and no separate per-language model: this is the same run,
# reported per language so a strong average cannot hide a collapsed language.
import gzip as _gzip
import json as _json
from pathlib import Path as _Path

import numpy as _np
import pandas as _pd

_LANGUAGE_CACHE = {}
LANGUAGE_BREAKDOWN_ROWS = []


def _resolve_codes_file():
    for root in (_Path("/kaggle/input"), _Path("/kaggle/working"), _Path(".")):
        if not root.exists():
            continue
        for name in ("codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp"):
            for path in root.rglob(name):
                if path.is_file():
                    return path
    return None


def _open_any(path):
    with open(path, "rb") as probe:
        packed = probe.read(2) == b"\x1f\x8b"
    return _gzip.open(path, "rt", encoding="utf-8") if packed else open(path, "r", encoding="utf-8")


def code_languages():
    """``code_id -> language`` from the attached clean-data bundle."""
    if _LANGUAGE_CACHE:
        return _LANGUAGE_CACHE
    path = _resolve_codes_file()
    if path is None:
        print("[language-breakdown] codes.jsonl not found; breakdown will be skipped.")
        return _LANGUAGE_CACHE
    with _open_any(path) as stream:
        for line in stream:
            if not line.strip():
                continue
            record = _json.loads(line)
            code_id = str(record.get("code_id", record.get("id", record.get("idx", ""))))
            _LANGUAGE_CACHE[code_id] = str(record.get("language", record.get("lang", "unknown")))
    print(f"[language-breakdown] languages loaded for {len(_LANGUAGE_CACHE):,} codes.")
    return _LANGUAGE_CACHE


def _binary_scores(labels, predicted):
    labels = _np.asarray(labels, dtype=_np.int64)
    predicted = _np.asarray(predicted, dtype=_np.int64)
    tp = int(((predicted == 1) & (labels == 1)).sum())
    fp = int(((predicted == 1) & (labels == 0)).sum())
    tn = int(((predicted == 0) & (labels == 0)).sum())
    fn = int(((predicted == 0) & (labels == 1)).sum())
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return {
        "P": precision, "R": recall, "F1": f1,
        "Acc": (tp + tn) / max(1, len(labels)),
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "Pairs": int(len(labels)), "Positives": int((labels == 1).sum()),
    }


def record_language_breakdown(frame, scores, threshold, *, dataset, method, graph_type=None):
    """Partition this run's test predictions by pair language and record them."""
    languages = code_languages()
    if not languages or frame is None or not len(frame):
        return []
    scores = _np.asarray(scores, dtype=_np.float64).reshape(-1)
    labels = _np.asarray(frame["label"], dtype=_np.int64).reshape(-1)
    if len(scores) != len(labels):
        print(f"[language-breakdown] skipped {method}: {len(scores)} scores vs {len(labels)} labels.")
        return []
    predicted = (scores >= float(threshold)).astype(_np.int64)

    left = [languages.get(str(value), "unknown") for value in frame["left_id"]]
    right = [languages.get(str(value), "unknown") for value in frame["right_id"]]
    # Cross-language pairs get their own bucket instead of being attributed to
    # one side; ATCoder is entirely java<->python and would otherwise vanish.
    keys = [a if a == b else f"{min(a, b)}->{max(a, b)}" for a, b in zip(left, right)]

    rows = []
    for key in sorted(set(keys)):
        mask = _np.asarray([value == key for value in keys])
        row = {"Dataset": dataset, "Method": method, "GraphType": graph_type or "", "Language": key}
        row.update(_binary_scores(labels[mask], predicted[mask]))
        row["Threshold"] = float(threshold)
        rows.append(row)
    overall = {"Dataset": dataset, "Method": method, "GraphType": graph_type or "", "Language": "ALL"}
    overall.update(_binary_scores(labels, predicted))
    overall["Threshold"] = float(threshold)
    rows.append(overall)

    LANGUAGE_BREAKDOWN_ROWS.extend(rows)
    table = _pd.DataFrame(LANGUAGE_BREAKDOWN_ROWS)
    out_path = _Path("/kaggle/working") / f"{dataset}_language_breakdown.csv"
    try:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        table.to_csv(out_path, index=False)
    except OSError:
        out_path = _Path(f"{dataset}_language_breakdown.csv")
        table.to_csv(out_path, index=False)
    print(f"\n[language-breakdown] {method}{'/' + graph_type if graph_type else ''}")
    print(_pd.DataFrame(rows)[["Language", "P", "R", "F1", "Acc", "Pairs", "Positives"]].to_string(index=False))
    print(f"[language-breakdown] written to {out_path}")
    return rows

DATASET_KEY_FOR_BREAKDOWN = "semanticclonebench_v3"


# SemanticCloneBench V3 Deckard Baseline

Kaggle-ready Deckard-style no-train baseline. This is a self-contained characteristic-vector reproduction for SemanticCloneBench V3: it hashes token, token-bigram, and simple structural features, tunes the cosine threshold on valid, and reports test metrics.


In [ ]:
# Executes the unchanged baseline pipeline once per dataset in isolated state.
# This run writes SemanticCloneBench V3-only CSV files, for example xglue4_*_results.csv.
def run_one_dataset(dataset_key: str):
    from pathlib import Path
    import time

    # End-to-end method runtime: loading + preprocessing + training + evaluation.
    run_started = time.perf_counter()

    DATASET_KEY = dataset_key
    KAGGLE_DATA_ROOT = Path("/kaggle/input") / DATASET_KEY
    WORK_DIR = Path("/kaggle/working")
    SEED = 42

    MAX_TRAIN_PAIRS = RUN_CONFIG["max_train_pairs"]
    MAX_VALID_PAIRS = RUN_CONFIG["max_valid_pairs"]
    MAX_TEST_PAIRS = RUN_CONFIG["max_test_pairs"]

    HASH_DIM = 4096
    RESULTS_PATH = WORK_DIR / f"{DATASET_KEY}_deckard_baseline_results.csv"
    SCORES_PATH = WORK_DIR / f"{DATASET_KEY}_deckard_valid_threshold.csv"

    from pathlib import Path
    import gc
    import gzip
    import json
    import math
    import os
    import random
    import re
    import zipfile

    import numpy as np
    import pandas as pd
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    from tqdm.auto import tqdm


    def seed_everything(seed: int = 42):
        random.seed(seed)
        np.random.seed(seed)
        try:
            import torch
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
        except Exception:
            pass


    def is_gzip_file(path: Path) -> bool:
        try:
            with path.open("rb") as f:
                return f.read(2) == b"\x1f\x8b"
        except OSError:
            return False


    def open_text(path: Path):
        return gzip.open(path, "rt", encoding="utf-8") if is_gzip_file(path) else path.open("r", encoding="utf-8")


    def candidate_roots():
        roots = [KAGGLE_DATA_ROOT, Path("/kaggle/input")]
        return [r for r in roots if r.exists()]


    def resolve_file_path(path: Path, *names: str) -> Path:
        if path.is_file():
            return path
        if path.is_dir():
            direct = [path / name for name in names if (path / name).is_file()]
            if direct:
                return direct[0]
            matches = []
            for name in names:
                matches.extend(path.rglob(name))
            matches = [m for m in matches if m.is_file()]
            if matches:
                return sorted(matches, key=lambda p: (len(p.relative_to(path).parts), len(str(p))))[0]
        return path


    def find_file(*names: str) -> Path:
        matches = []
        for root in candidate_roots():
            for name in names:
                direct = root / name
                if direct.is_file():
                    matches.append(direct)
                elif direct.is_dir():
                    resolved = resolve_file_path(direct, *names)
                    if resolved.is_file():
                        matches.append(resolved)
                matches.extend([p for p in root.rglob(name) if p.is_file()])
        if not matches:
            seen = []
            for root in candidate_roots():
                seen.extend(str(p) for p in sorted(root.rglob("*"))[:30])
            raise FileNotFoundError(f"Could not find {names}. First available paths: {seen}")
        return sorted(matches, key=lambda p: (len(p.parts), len(p.name), str(p)))[0]


    def load_pairs(path: Path) -> pd.DataFrame:
        path = resolve_file_path(path, "pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
        compression = "gzip" if is_gzip_file(path) else None
        df = pd.read_csv(path, compression=compression, dtype={"left_id": str, "right_id": str, "split": str, "label": np.int64})
        df["left_id"] = df["left_id"].astype(str)
        df["right_id"] = df["right_id"].astype(str)
        return df


    def load_codes(path: Path) -> dict[str, str]:
        path = resolve_file_path(path, "codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp")
        codes = {}
        with open_text(path) as f:
            for line in tqdm(f, desc="Loading codes", unit="code"):
                if not line.strip():
                    continue
                obj = json.loads(line)
                codes[str(obj["code_id"])] = obj["code"]
        return codes


    def maybe_limit_split(df: pd.DataFrame, split: str, max_rows: int | None, seed: int) -> pd.DataFrame:
        part = df[df["split"] == split].copy()
        if max_rows is not None and len(part) > max_rows:
            part = part.sample(n=max_rows, random_state=seed)
        return part.reset_index(drop=True)


    def metric_dict(labels, scores, threshold: float) -> dict:
        pred = (np.asarray(scores) >= threshold).astype(np.int64)
        labels = np.asarray(labels).astype(np.int64)
        p, r, f1, _ = precision_recall_fscore_support(labels, pred, average="binary", zero_division=0)
        acc = accuracy_score(labels, pred)
        tp = int(((pred == 1) & (labels == 1)).sum())
        fp = int(((pred == 1) & (labels == 0)).sum())
        tn = int(((pred == 0) & (labels == 0)).sum())
        fn = int(((pred == 0) & (labels == 1)).sum())
        return {"P": p, "R": r, "F1": f1, "Acc": acc, "TP": tp, "FP": fp, "TN": tn, "FN": fn}


    def choose_threshold(labels, scores, n_grid: int = 401) -> tuple[float, dict]:
        labels = np.asarray(labels).astype(np.int64)
        scores = np.asarray(scores, dtype=np.float32)
        if len(scores) == 0:
            return 0.5, metric_dict(labels, scores, 0.5)
        qs = np.linspace(0.0, 1.0, n_grid)
        thresholds = np.unique(np.quantile(scores, qs))
        thresholds = np.unique(np.concatenate([thresholds, np.array([0.5], dtype=np.float32)]))
        best_thr = float(thresholds[0])
        best = None
        for thr in thresholds:
            m = metric_dict(labels, scores, float(thr))
            if best is None or (m["F1"], m["Acc"]) > (best["F1"], best["Acc"]):
                best = m
                best_thr = float(thr)
        return best_thr, best


    seed_everything(SEED)
    codes_path = find_file("codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp")
    pairs_path = find_file("pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
    print("codes:", codes_path, "is_file=", codes_path.is_file())
    print("pairs:", pairs_path, "is_file=", pairs_path.is_file())
    codes = load_codes(codes_path)
    pairs_df = load_pairs(pairs_path)
    train_df = maybe_limit_split(pairs_df, "train", MAX_TRAIN_PAIRS, SEED)
    valid_df = maybe_limit_split(pairs_df, "valid", MAX_VALID_PAIRS, SEED + 1)
    test_df = maybe_limit_split(pairs_df, "test", MAX_TEST_PAIRS, SEED + 2)
    print("pairs:")
    print(pairs_df.groupby(["split", "label"]).size())
    print(f"using train/valid/test={len(train_df):,}/{len(valid_df):,}/{len(test_df):,}")


    TOKEN_RE = re.compile(r"[A-Za-z_]\w*|\d+|==|!=|<=|>=|&&|\|\||[{}()\[\];,.:+\-*/%<>=!&|^~?]")
    KEYWORDS = {
        "if", "else", "for", "while", "do", "switch", "case", "return", "break", "continue",
        "try", "catch", "finally", "throw", "throws", "new", "class", "interface", "enum",
        "public", "private", "protected", "static", "final", "void", "int", "long", "float",
        "double", "boolean", "char", "byte", "short", "String"
    }


    def add_hash(vec, key: str, value: float = 1.0):
        idx = hash(key) % HASH_DIM
        vec[idx] += value


    def characteristic_vector(code: str) -> np.ndarray:
        toks = TOKEN_RE.findall(code)
        vec = np.zeros(HASH_DIM, dtype=np.float32)
        for tok in toks:
            norm = "ID" if re.match(r"[A-Za-z_]\w*$", tok) and tok not in KEYWORDS else tok
            norm = "NUM" if tok.isdigit() else norm
            add_hash(vec, "tok:" + norm)
        for a, b in zip(toks, toks[1:]):
            aa = "ID" if re.match(r"[A-Za-z_]\w*$", a) and a not in KEYWORDS else a
            bb = "ID" if re.match(r"[A-Za-z_]\w*$", b) and b not in KEYWORDS else b
            add_hash(vec, "bi:" + aa + "|" + bb, 0.5)
        depth = 0
        max_depth = 0
        for tok in toks:
            if tok in "{([":
                depth += 1
                max_depth = max(max_depth, depth)
            elif tok in "})]":
                depth = max(0, depth - 1)
        scalar = {
            "len_bucket": int(math.log2(len(toks) + 1)),
            "line_bucket": int(math.log2(code.count("\n") + 2)),
            "max_depth": max_depth,
            "returns": code.count("return"),
            "branches": sum(code.count(k) for k in ["if", "switch", "case"]),
            "loops": sum(code.count(k) for k in ["for", "while", "do"]),
        }
        for key, value in scalar.items():
            add_hash(vec, f"{key}:{value}", 2.0)
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec


    needed_ids = set(train_df.left_id) | set(train_df.right_id) | set(valid_df.left_id) | set(valid_df.right_id) | set(test_df.left_id) | set(test_df.right_id)
    code_ids = sorted(needed_ids)
    id_to_row = {cid: i for i, cid in enumerate(code_ids)}
    vectors = np.stack([characteristic_vector(codes.get(cid, "")) for cid in tqdm(code_ids, desc="Deckard vectors")])
    print("codes=", len(code_ids), "dim=", vectors.shape[1])


    def score_pairs(df: pd.DataFrame, batch_size: int = 250_000) -> np.ndarray:
        out = np.empty(len(df), dtype=np.float32)
        left_all = df["left_id"].map(id_to_row).to_numpy(np.int64)
        right_all = df["right_id"].map(id_to_row).to_numpy(np.int64)
        for start in tqdm(range(0, len(df), batch_size), desc="Scoring pairs"):
            end = min(start + batch_size, len(df))
            out[start:end] = (vectors[left_all[start:end]] * vectors[right_all[start:end]]).sum(axis=1)
        return out


    valid_scores = score_pairs(valid_df)
    threshold, valid_metrics = choose_threshold(valid_df["label"].to_numpy(), valid_scores)
    test_scores = score_pairs(test_df)
    test_metrics = metric_dict(test_df["label"].to_numpy(), test_scores, threshold)
    record_language_breakdown(test_df, test_scores, threshold, dataset=DATASET_KEY_FOR_BREAKDOWN, method="Deckard")

    row = {
        "Method": "Deckard",
        "BestEpoch": "",
        "BestValidF1": valid_metrics["F1"],
        **test_metrics,
        "Threshold": threshold,
        "TrainPairs": len(train_df),
        "ValidPairs": len(valid_df),
        "TestPairs": len(test_df),
    }
    # Deckard is non-neural: parameter count is explicitly zero.
    row["TrainableParameters"] = 0
    row["RuntimeSeconds"] = float(time.perf_counter() - run_started)
    row["RuntimeMinutes"] = row["RuntimeSeconds"] / 60.0
    pd.DataFrame([row]).to_csv(RESULTS_PATH, index=False)
    pd.DataFrame([{"Threshold": threshold, **valid_metrics}]).to_csv(SCORES_PATH, index=False)
    print(pd.DataFrame([row]))
    print("saved:", RESULTS_PATH)

    # Research-reproducibility manifest and enriched result table.
    # This is deliberately written after evaluation so measured runtime is final.
    import datetime as _datetime
    try:
        _gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        _gpu_capability = ".".join(map(str, torch.cuda.get_device_capability(0))) if torch.cuda.is_available() else None
        _torch_version = torch.__version__
    except Exception:
        _gpu_name, _gpu_capability, _torch_version = "unavailable", None, "unavailable"
    _completed_utc = _datetime.datetime.now(_datetime.timezone.utc).isoformat()
    _run_seconds = float(time.perf_counter() - run_started)
    _shared_fields = {
        "Dataset": DATASET_KEY,
        "RunProfile": RUN_PROFILE,
        "Seed": int(SEED),
        "ConfiguredEpochs": int(EPOCHS) if "EPOCHS" in locals() else None,
        "BatchSize": int(BATCH_SIZE) if "BATCH_SIZE" in locals() else None,
        "LearningRate": float(LEARNING_RATE) if "LEARNING_RATE" in locals() else None,
        "WeightDecay": float(WEIGHT_DECAY) if "WEIGHT_DECAY" in locals() else None,
        "GPU": _gpu_name,
        "GPUCapability": _gpu_capability,
        "TorchVersion": _torch_version,
        "CompletedUTC": _completed_utc,
    }
    if "results_df" in locals():
        _result_table = results_df.copy()
    elif "result" in locals():
        _result_table = pd.DataFrame([result])
    elif "row" in locals():
        _result_table = pd.DataFrame([row])
    else:
        _result_table = pd.DataFrame()
    for _field, _value in _shared_fields.items():
        _result_table[_field] = _value
    if "RuntimeSeconds" not in _result_table.columns:
        _result_table["RuntimeSeconds"] = _run_seconds
    if "RuntimeMinutes" not in _result_table.columns:
        _result_table["RuntimeMinutes"] = _run_seconds / 60.0
    _result_path = RESULTS_PATH if "RESULTS_PATH" in locals() else out_path
    _result_table.to_csv(_result_path, index=False)
    _metadata = {
        **_shared_fields,
        "RunLabel": RUN_LABEL,
        "RuntimeSeconds": _run_seconds,
        "RuntimeMinutes": _run_seconds / 60.0,
        "RequestedPairCaps": {
            "train": MAX_TRAIN_PAIRS,
            "valid": MAX_VALID_PAIRS,
            "test": MAX_TEST_PAIRS,
        },
        "ModelConfiguration": {
            _name: locals().get(_name)
            for _name in (
                "MAX_AST_NODES", "MAX_AST_EDGES", "MAX_STATEMENTS", "MAX_NODE_TYPES", "MAX_NODES",
                "EMBED_DIM", "HIDDEN_DIM", "TREE_HIDDEN_DIM", "CODE_DIM", "DROPOUT",
                "GRAPH_TYPE", "GRAPH_TYPES", "K_EIGEN", "USE_EIGEN_STATS", "USE_GRAPH_STATS",
            ) if _name in locals()
        },
        "OutputFiles": {
            "results": str(_result_path),
            "history": str(HISTORY_PATH) if "HISTORY_PATH" in locals() else (str(history_path) if "history_path" in locals() else None),
        },
    }
    _metadata_path = WORK_DIR / f"{DATASET_KEY}_{RUN_LABEL}_run_metadata.json"
    _metadata_path.write_text(json.dumps(_metadata, indent=2, default=str), encoding="utf-8")
    print("Research metadata:", _metadata_path)
    if "results_df" in locals():
        results_df = _result_table

    if "results_df" in locals():
        return results_df.copy()
    if "result" in locals():
        return pd.DataFrame([result])
    if "row" in locals():
        return pd.DataFrame([row])
    raise RuntimeError("The baseline did not produce a result table.")


from IPython.display import display
import pandas as pd

all_dataset_results = {}
for current_dataset_key in DATASET_KEYS:
    print("\n" + "=" * 96)
    print(f"Running {current_dataset_key.upper()}")
    print("=" * 96)
    dataset_results = run_one_dataset(current_dataset_key)
    # The per-run result table already carries Dataset for research metadata.
    # Preserve a single authoritative value rather than inserting a duplicate column.
    if "Dataset" in dataset_results.columns:
        dataset_results["Dataset"] = current_dataset_key.upper()
    else:
        dataset_results.insert(0, "Dataset", current_dataset_key.upper())
    all_dataset_results[current_dataset_key] = dataset_results

print("\n" + "=" * 96)
print("Final result tables")
print("=" * 96)
for current_dataset_key in DATASET_KEYS:
    print(f"\n{current_dataset_key.upper()} results")
    display(all_dataset_results[current_dataset_key])

combined_results = pd.concat(
    [all_dataset_results[key] for key in DATASET_KEYS],
    ignore_index=True,
)
combined_path = Path("/kaggle/working") / f"{RUN_LABEL}_combined_dataset_results.csv"
combined_results.to_csv(combined_path, index=False)
print("Combined results:", combined_path)
